# Phase 3 Session 2 — NB3: ABSA Training (with Retrieval)

**Goal:** Train ABSA with improved retrieval (top_k=2, threshold=0.3, plain CE, no CRF).

**Input:**
- `p3s2-embedding`: embedding checkpoint + processed data (from nb1)
- `p3s2-index`: FAISS index (from nb2)

**Config:** `absa.yaml` + `retrieval_v2.yaml`

**Baseline:** Phase 2 no-retrieval Joint F1 = 0.6104

**Output:** Upload `/kaggle/working/outputs_p3s2_nb3/` as Kaggle dataset `p3s2-absa` for nb4.

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
NB1_INPUT = None
for candidate in ['/kaggle/input/p3s2-embedding',
                  '/kaggle/input/datasets/lcminhc/p3s2-embedding']:
    if os.path.exists(candidate):
        NB1_INPUT = candidate
        break
NB2_INPUT = None
for candidate in ['/kaggle/input/p3s2-index',
                  '/kaggle/input/datasets/lcminhc/p3s2-index']:
    if os.path.exists(candidate):
        NB2_INPUT = candidate
        break
assert NB1_INPUT, 'Dataset p3s2-embedding not found'
assert NB2_INPUT, 'Dataset p3s2-index not found'
print(f'NB1 input: {NB1_INPUT}')
print(f'NB2 input: {NB2_INPUT}')

# Copy embedding checkpoint
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{NB1_INPUT}/embedding_best.pt', 'checkpoints/embedding/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding/best.pt") / 1e6:.1f} MB')

# Copy processed data (flat layout)
os.makedirs('data/processed', exist_ok=True)
for f in ['bio_tagging.jsonl', 'classification.jsonl', 'contrastive_triplets.jsonl']:
    shutil.copy(f'{NB1_INPUT}/{f}', f'data/processed/{f}')
    print(f'data/processed/{f}')

# Copy index (flat layout)
os.makedirs('indexes', exist_ok=True)
for f in os.listdir(NB2_INPUT):
    if f.endswith(('.faiss', '.jsonl', '.npy')):
        shutil.copy(f'{NB2_INPUT}/{f}', f'indexes/{f}')
        print(f'indexes/{f}')

## 1. Train ABSA with Retrieval

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

torch.cuda.empty_cache()
import gc
gc.collect()
print('GPU cache cleared.')

In [ ]:
!python scripts/04_train_absa.py \
    --config configs/absa.yaml \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --retrieval_config configs/retrieval_v2.yaml \
    --grad_accum_steps 8 \
    --patience 999

In [ ]:
if os.path.exists('logs/absa_training.jsonl'):
    print(f'{"Epoch":<8} {"Loss":<10} {"Span F1":<10} {"Sent Acc":<10} {"Joint F1":<10}')
    print('-' * 48)
    with open('logs/absa_training.jsonl') as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<10.4f} {rec['span_f1']:<10.4f} "
                  f"{rec['sentiment_acc']:<10.4f} {rec['joint_f1']:<10.4f}")

## 2. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p3s2_nb3'
os.makedirs(output_dir, exist_ok=True)

# ABSA checkpoint
if os.path.exists('checkpoints/absa/best.pt'):
    shutil.copy('checkpoints/absa/best.pt',
                os.path.join(output_dir, 'absa_best.pt'))
    size_mb = os.path.getsize('checkpoints/absa/best.pt') / 1e6
    print(f'absa_best.pt: {size_mb:.1f} MB')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(output_dir, 'logs'), dirs_exist_ok=True)

print(f'\nOutputs saved to {output_dir}')
print('Upload this as Kaggle dataset: p3s2-absa')

In [ ]:
shutil.make_archive(
    base_name="/kaggle/working/outputs_p3s2_nb3_backup",
    format="zip",
    root_dir="/kaggle/working",
    base_dir="outputs_p3s2_nb3"
)
size_mb = os.path.getsize("/kaggle/working/outputs_p3s2_nb3_backup.zip") / 1e6
print(f"outputs_p3s2_nb3_backup.zip: {size_mb:.1f} MB")